# Webizen AI Synth — Google Colab Ultra Heavy-Compute Burst Worker
**Copyright (c) 2026 Timothy Charles Holborn** | **CC BY-NC-ND 4.0**

This notebook turns a **Google Colab Ultra** runtime (**NVIDIA A100 40GB/80GB** or **NVIDIA L4 24GB** + **83 GB System RAM**) into an on-demand remote heavy-compute burst worker for **Webizen AI Synth** (`synth_colab` / `synth colab`).

### Supported Routines & Model Decision Guide:
1. `trellis2_image_to_3d` — `microsoft/TRELLIS.2-4B` (MIT) single-image to 3D O-Voxel ($512^3$–$1536^3$) + 4K PBR `.glb` + `.ovoxel.json` (`M64HoloRecord` clusters + `VOX0` DAG nodes). Best for complex topology, internal cavities, foliage, and perforated anatomy.
2. `trellis2_multiview_to_3d` — Multi-view (2..4 images) conditioned 3D O-Voxel + 4K PBR `.glb` + `.ovoxel.json`.
3. `hunyuan3d_shape_and_pbr` — `tencent/Hunyuan3D-2.1` (Apache-2.0) decoupled DiT shape generation & delighted 4K PBR texture baking (`mode="pbr_only"` bakes delighted PBR maps onto an existing Synth `.vibe` mesh).
4. `partcrafter_decompose_3d` — `wgsxm/PartCrafter` (MIT) compositional multi-part 3D decomposition into a multi-node `.glb` (`output.parts.json`) for Synth Mode-A Archetype `.10d` + Mode-B `instances/*.10d` (`AREF`/`INST`).
5. `unirig_auto_skeleton` — `VAST-AI-Research/UniRig` (MIT) autoregressive skeletal bone tree & skinning weights (`output.skeleton.json` + rigged `.glb`) for `.10d` `SKL1`/`SKL2`.
6. `ontology_compile_q42` — QualiaDB / Synth CLI High-RAM (83 GB) online ontology fetch (`source_url` or BioPortal `acronym`), OWL refactoring (`skos:Concept` refs + SHACL shapes + N3Logic rules), and `.q42` volume compilation.

In [ ]:
# CELL 1: Verify GPU (A100 / L4 VRAM Check) & Mount Google Drive Persistent Cache
import os, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Synth-Colab-Ultra')
CACHE_ROOT = DRIVE_ROOT / 'cache'
WHEELS_CACHE = CACHE_ROOT / 'wheels'
HF_CACHE = CACHE_ROOT / 'huggingface'
BIN_CACHE = CACHE_ROOT / 'bin'
for d in (WHEELS_CACHE, HF_CACHE, BIN_CACHE, DRIVE_ROOT / 'queue' / 'pending', DRIVE_ROOT / 'queue' / 'running', DRIVE_ROOT / 'queue' / 'completed', Path('/content/scratch')):
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['QUALIA_INGEST_SCRATCH'] = '/content/scratch'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode())

In [ ]:
# CELL 2: Bootstrap & Cache 3D Models (TRELLIS.2, Hunyuan3D-2.1, PartCrafter, UniRig) + QualiaDB/Synth Linux CLI on Google Drive
import os, shutil, subprocess, sys
from pathlib import Path

TRELLIS2_REPO = Path('/content/TRELLIS.2')
if not TRELLIS2_REPO.exists():
    subprocess.run(['git', 'clone', '--recursive', 'https://github.com/microsoft/TRELLIS.2.git', str(TRELLIS2_REPO)], check=True)

cached_wheels = list(Path('/content/drive/MyDrive/Synth-Colab-Ultra/cache/wheels').glob('*.whl'))
if cached_wheels:
    print(f'Installing {len(cached_wheels)} pre-built CUDA wheels from Google Drive cache...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps'] + [str(w) for w in cached_wheels], check=False)
else:
    print('First-time setup: building TRELLIS.2 CUDA extensions and caching wheels to Google Drive...')
    subprocess.run(['bash', 'setup.sh', '--basic', '--flash-attn', '--nvdiffrast', '--cumesh', '--o-voxel', '--flexgemm'], cwd=str(TRELLIS2_REPO), check=False)

# Provision QualiaDB / Synth Linux CLI binary for online ontology -> .q42 compilation
SYNTH_CACHE_BIN = Path('/content/drive/MyDrive/Synth-Colab-Ultra/cache/bin/synth')
if SYNTH_CACHE_BIN.is_file():
    os.chmod(SYNTH_CACHE_BIN, 0o755)
    print(f'QualiaDB/Synth Linux CLI ready from Drive cache: {SYNTH_CACHE_BIN}')
elif Path('/content/synth/Cargo.toml').is_file():
    print('Building QualiaDB/Synth Linux CLI into Drive cache...')
    subprocess.run(['cargo', 'build', '--release', '--bin', 'synth'], cwd='/content/synth', check=True)
    shutil.copy2('/content/synth/target/release/synth', SYNTH_CACHE_BIN)
    os.chmod(SYNTH_CACHE_BIN, 0o755)

sys.path.insert(0, str(TRELLIS2_REPO))

In [ ]:
# CELL 3: Configure Bearer Token & Idle Auto-Disconnect Watchdog (Protects Google AI Ultra Compute Units)
import os, secrets
from google.colab import userdata

try:
    token = userdata.get('SYNTH_COLAB_TOKEN')
except Exception:
    token = None

if not token:
    token = secrets.token_urlsafe(32)
    print('Generated ephemeral SYNTH_COLAB_TOKEN (save in Colab Secrets as SYNTH_COLAB_TOKEN for persistence).')

os.environ['SYNTH_COLAB_TOKEN'] = token
IDLE_TIMEOUT_MINUTES = 30  # Automatically calls google.colab.runtime.unassign() after 30m idle
print(f'Bearer auth active. Idle auto-disconnect watchdog set to {IDLE_TIMEOUT_MINUTES} minutes.')

In [ ]:
# CELL 4: Start the Synth Colab Dual-Transport Worker (HTTP + Google Drive Queue)
# Note: On your local Windows machine, configure the token securely via --token-stdin or SYNTH_COLAB_TOKEN:
#   $env:SYNTH_COLAB_TOKEN = '<YOUR_TOKEN>'
#   synth colab configure --url https://<TUNNEL_HOST> --drive-dir "G:\My Drive\Synth-Colab-Ultra"
import colab_worker
colab_worker.run_server(
    port=8099,
    work_root='/content/synth_colab_work',
    drive_root='/content/drive/MyDrive/Synth-Colab-Ultra',
    idle_timeout_minutes=IDLE_TIMEOUT_MINUTES,
)